## Workflow Overview  
**Anomalous Sightings Archive Project**

### Tools & Environment
- **Version Control**: GitHub
- **IDE**: VS Code
- **Primary Development**: Jupyter Notebooks (data wrangling, database creation, and visualizations)
- **Modular Python Scripts** (in `python/`):
    - Weather API integration
    - KP Index API fetching and CSV generation
    - Geohash generation from latitude/longitude
    - Proximity table computation and distance enrichment

### Project Workflow

1. **Data Ingestion**  
   Load original datasets into the `data/` directory:  
   - Bigfoot reports (2 BFRO datasets from Kaggle)  
   - UAP reports (NUFORC dataset from Kaggle)  
   - US Census 2010 state population data  
   - Generate `kp_index.csv` via dedicated API script

2. **Data Cleaning** (Pandas)  
   - Standard cleaning and normalization  
   - Merge the two Bigfoot datasets into `combined_bigfoot.csv`  
   - Save cleaned DataFrames as CSVs for backup and reproducibility

3. **Data Enrichment** (Pandas)  
   - Add solar KP Index and AP Index to relevant DataFrames  
   - Apply historical weather data (2010–2014 UAP reports) using `weather_api.py`  
     *(Full UAP dataset spans 1940–2014; Bigfoot data already contains weather)*  
   - Create proximity table by merging UAP and Bigfoot records on `geohash_7`  
   - Reorder columns to align with the Entity Relationship Diagram (ERD)

4. **Relational Database Creation** (SQLite)  
   Build the database with the following tables:  
   - `bigfoot_reports` (PK: `bf_id`)  
   - `uap_reports` (PK: `uap_id`)  
   - `states` (PK: `state_code`)  
   - `proximity` (composite PK: `bf_id` + `uap_id`)  
   - `kp_index` (PK: `datetime`)

5. **Analysis**  
   Execute SQL queries against the SQLite database to generate insights for visualization.

6. **Visualizations**  
   - Charts: Matplotlib + Seaborn  
   - Maps: GeoPandas + Folium

7. **Front-End & Stretch Goals**  
   - Interactive dashboard using **Streamlit** (primary option) or a static site  
   - User experience submission form (Python script / Streamlit page)

In [8]:
# imports 

import pandas as pd
import datetime
import pygeohash as pgh
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import sys
import time 
import requests
import dotenv
from pathlib import Path
import sqlite3

from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv


root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.geo_location import create_geohashes

In [9]:
# uap with weather - remove index

uap_2010_wx_df = pd.read_csv("../data/processed/sighting_with_weather_v2 copy.csv")
uap_2010_wx_df = uap_2010_wx_df.drop(columns='Unnamed: 0')
uap_2010_wx_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups
0,2010-10-10 01:00:00,orchard park,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear
1,2010-10-10 02:30:00,harrisburg,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear
2,2010-10-10 03:00:00,euclid,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear
3,2010-10-10 08:30:00,starr,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear
4,2010-10-10 10:45:00,leominster,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear


In [10]:
us_uap_1940_df = pd.read_csv("../data/processed/us_uap_1940_v1.csv")

uap_2010_wx_df = pd.merge(
    uap_2010_wx_df,
    us_uap_1940_df[["uap_id", "state_code", "datetime", "city"]],
    on=["datetime", "city"],
    how="left"  
)

uap_2010_wx_df["uap_id"] = uap_2010_wx_df["uap_id"].astype('Int64')

uap_2010_wx_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups,uap_id,state_code
0,2010-10-10 01:00:00,orchard park,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear,49443,NY
1,2010-10-10 02:30:00,harrisburg,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear,49444,PA
2,2010-10-10 03:00:00,euclid,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear,49445,OH
3,2010-10-10 08:30:00,starr,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear,49446,SC
4,2010-10-10 10:45:00,leominster,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear,49447,MA


In [11]:
uap_2010_wx_df = uap_2010_wx_df[[
    'uap_id','datetime', 'city', 'state_code', 'state', 'country', 
    'shape', 'duration (seconds)', 'duration (hours/min)', 'comments', 'date posted', 
    'latitude','longitude', 'rounded_dt', 'date_str', 'hour_str', 
    'weather_cloud','weather_temp_f', 'weather_condition', 'condition_groups'
    ]]

uap_2010_wx_df.head()

,uap_id,datetime,city,state_code,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups
0,49443,2010-10-10 01:00:00,orchard park,NY,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear
1,49444,2010-10-10 02:30:00,harrisburg,PA,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear
2,49445,2010-10-10 03:00:00,euclid,OH,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear
3,49446,2010-10-10 08:30:00,starr,SC,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear
4,49447,2010-10-10 10:45:00,leominster,MA,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear


In [12]:
uap_2010_wx_df.to_csv("../data/final/us_uap_2010_2014_weather.csv", index=False)

In [20]:
"""
add renamed csvs to final/ 

us_bigfoot_reports
us_uap_reports
proximity
states
"""

df = pd.read_csv("../data/processed/combined_bigfoot_v1.csv")
df.to_csv("../data/final/bigfoot_reports.csv", index=False)

df = pd.read_csv("../data/processed/us_uap_1940_v1.csv")
df.to_csv("../data/final/uap_reports.csv", index=False)

df = pd.read_csv("../data/processed/proximity_v1.csv")
df.to_csv("../data/final/proximity.csv", index=False)

df = pd.read_csv("../data/processed/states_2010_population_data.csv")
df.to_csv("../data/final/states.csv", index=False)

In [ ]:
# Paths to Database and Folder with Processed CSV files
db_path = Path("../data/sql/")

# Connection 
connection = sqlite3.connect(db_path / "anomalous_sightings.db")

In [ ]:
# UAP Reports Table (All US Reports 1940 - 2014)

uap_df = pd.read_csv("../data/final/uap_reports.csv")

uap_df.to_sql(
    name='uap_reports', 
    con=connection, 
    if_exists='replace', 
    index=True,         
    dtype={
        'uap_id': 'INTEGER PRIMARY KEY AUTOINCREMENT', 
        'datetime': 'TEXT',
        'city': 'TEXT',
        'state_code': 'TEXT',
        'state': 'TEXT',
        'duration_secs': 'INTEGER',
        'latitude': 'REAL',
        'longitude': 'REAL',
        'geohash_5': 'TEXT',
        'geohash_6': 'TEXT',
        'geohash_7': 'TEXT',
        'full_date': 'TEXT',
        'year': 'INTEGER',
        'month': 'INTEGER',
        'season': 'TEXT',
        'datetime_formatted': 'TEXT',
        'shape': 'TEXT',
        'shape_group': 'TEXT',
        'comments': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)

# Bigfoot Reports Table (All US Reports 1950 - 2021)

bigfoot_df = pd.read_csv("../data/final/bigfoot_reports.csv")

bigfoot_df.to_sql(
    name='bigfoot_reports', 
    con=connection, 
    if_exists='replace', 
    index=True,         
    dtype={
        'bf_id': 'INTEGER PRIMARY KEY AUTOINCREMENT', 
        'full_date': 'TEXT', 
        'title': 'TEXT', 
        'state_code': 'TEXT', 
        'state': 'TEXT', 
        'latitude': 'REAL', 
        'longitude': 'REAL', 
        'geohash_5': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7': 'TEXT', 
        'geohash': 'TEXT', 
        'date': 'TEXT', 
        'year': 'INTEGER', 
        'month': 'INTEGER', 
        'day': 'INTEGER', 
        'season': 'TEXT', 
        'temperature_mid': 'REAL', 
        'dew_point': 'REAL', 
        'cloud_cover': 'REAL', 
        'moon_phase': 'REAL', 
        'precip_type': 'TEXT', 
        'classification': 'TEXT', 
        'observed': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)

# States Table

states_df = pd.read_csv("../data/final/states.csv")

states_df.to_sql(
    name='states', 
    con=connection, 
    if_exists='replace', 
    index=True,         
    dtype={
        'state_code': 'TEXT PRIMARY KEY',
        'state': 'TEXT', 
        '2010_population': 'INTEGER' 
    }
)

# Proximity Table

proximity_df = pd.read_csv("../data/final/proximity.csv")

proximity_df.to_sql(
    name='proximity', 
    con=connection, 
    if_exists='replace', 
    index=True,         
    dtype={
        'bf_id': 'INTEGER', 
        'uap_id': 'INTEGER', 
        'city': 'TEXT', 
        'state_code': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7_bf': 'TEXT', 
        'geohash_7_uap': 'TEXT', 
        'latitude_bf': 'REAL', 
        'longitude_bf': 'REAL', 
        'latitude_uap': 'REAL', 
        'longitude_uap': 'REAL', 
        'full_date_bf': 'TEXT', 
        'full_date_uap': 'TEXT', 
        'date_diff_days': 'INTEGER',
        'year_bf': 'INTEGER', 
        'year_uap': 'INTEGER', 
        'distance_meters': 'REAL', 
        'proximity_score': 'REAL',
        'proximity_rank': 'INTEGER' 
    }
)




1430

In [29]:
proximity_df = pd.read_csv("../data/final/proximity.csv")
proximity_df.columns


Index(['bf_id', 'uap_id', 'city', 'state_code', 'geohash_6', 'geohash_7_bf',
       'geohash_7_uap', 'latitude_bf', 'longitude_bf', 'latitude_uap',
       'longitude_uap', 'full_date_bf', 'full_date_uap', 'date_diff_days',
       'year_bf', 'year_uap', 'distance_meters', 'proximity_score',
       'proximity_rank'],
      dtype='object')

In [16]:
# verify tables

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)

,name
0,bigfoot_reports
1,proximity
2,states
3,uap_reports
4,us_uap_2010_2014_weather
